In [1]:
from googleapiclient.discovery import build
from google.colab import userdata
import pandas as pd
import numpy as np
from datetime import datetime
import time

In [2]:
youtube = build('youtube', 'v3', developerKey= userdata.get('my_yt_v3_api_key'))

In [3]:
# Disclaimer: Please note that these IDs belong to original YouTube channels and are being used based on the conducted survey for educational purposes only.

CHANNEL_IDS = [
    "UC0T6MVd3wQDB5ICAe45OxaQ",  # WsCubeTech
    "UC4o8Fdpv3g_AjgShAeivqpA",  # Naresh IT
    "UC640y4UvDAlya_WOj5U4pfA",  # NPTEL
    "UCCWi3hpnq_Pe03nGxuS7isg",  # CampusX
    "UCBwmMxybNva6P_5VmxjzwqA",  # Apna College
    "UCeVMnSShP_Iviwkknt83cww",  # CodeWithHarry
    "UCNU_lfiiWBdtULKOw6X0Dig",  # Krish Naik
    "UCM-yUTYGmrNvKOCcAl21g3w",  # Jenny's Lectures
    "UC-ZZjHr4nl5t32pFl3Sqf0A",  # Genie Ashwani
    "UCf9T51_FmMlfhiGpoes0yFA"   # Piyush Garg
]

In [4]:
# Get channel's upload id

upload_playlist_ids = {}

for channel_id in CHANNEL_IDS:
    response = youtube.channels().list(part="contentDetails",id=channel_id).execute()
    upload_playlist_ids[channel_id] = (response["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"])

print(upload_playlist_ids)

{'UC0T6MVd3wQDB5ICAe45OxaQ': 'UU0T6MVd3wQDB5ICAe45OxaQ', 'UC4o8Fdpv3g_AjgShAeivqpA': 'UU4o8Fdpv3g_AjgShAeivqpA', 'UC640y4UvDAlya_WOj5U4pfA': 'UU640y4UvDAlya_WOj5U4pfA', 'UCCWi3hpnq_Pe03nGxuS7isg': 'UUCWi3hpnq_Pe03nGxuS7isg', 'UCBwmMxybNva6P_5VmxjzwqA': 'UUBwmMxybNva6P_5VmxjzwqA', 'UCeVMnSShP_Iviwkknt83cww': 'UUeVMnSShP_Iviwkknt83cww', 'UCNU_lfiiWBdtULKOw6X0Dig': 'UUNU_lfiiWBdtULKOw6X0Dig', 'UCM-yUTYGmrNvKOCcAl21g3w': 'UUM-yUTYGmrNvKOCcAl21g3w', 'UC-ZZjHr4nl5t32pFl3Sqf0A': 'UU-ZZjHr4nl5t32pFl3Sqf0A', 'UCf9T51_FmMlfhiGpoes0yFA': 'UUf9T51_FmMlfhiGpoes0yFA'}


In [5]:
# Get Subscriber Count (ONCE PER CHANNEL)

subscriber_counts = {}

for channel_id in CHANNEL_IDS:
    response = youtube.channels().list(part="statistics",id=channel_id).execute()
    subscriber_counts[channel_id] = int(response["items"][0]["statistics"].get("subscriberCount", 0))

print(subscriber_counts)

{'UC0T6MVd3wQDB5ICAe45OxaQ': 4220000, 'UC4o8Fdpv3g_AjgShAeivqpA': 1330000, 'UC640y4UvDAlya_WOj5U4pfA': 2200000, 'UCCWi3hpnq_Pe03nGxuS7isg': 459000, 'UCBwmMxybNva6P_5VmxjzwqA': 7340000, 'UCeVMnSShP_Iviwkknt83cww': 9330000, 'UCNU_lfiiWBdtULKOw6X0Dig': 1340000, 'UCM-yUTYGmrNvKOCcAl21g3w': 2030000, 'UC-ZZjHr4nl5t32pFl3Sqf0A': 145000, 'UCf9T51_FmMlfhiGpoes0yFA': 347000}


In [6]:
# Fetch Videos from Upload Playlists (Pagination Safe)

MAX_VIDEOS_PER_CHANNEL = 300

video_rows = []

for channel_id, playlist_id in upload_playlist_ids.items():

    next_page_token = None
    video_count = 0

    while True:
        request = youtube.playlistItems().list(part="snippet",playlistId=playlist_id,maxResults=50,pageToken=next_page_token)
        response = request.execute()

        for item in response["items"]:
            if video_count >= MAX_VIDEOS_PER_CHANNEL:
                break

            snippet = item["snippet"]

            video_rows.append({
                "channel_id": channel_id,
                "video_id": snippet["resourceId"]["videoId"],
                "video_title": snippet["title"],
                "description": snippet["description"],
                "published_at": snippet["publishedAt"],
                "subscriber_count": subscriber_counts[channel_id]
            })

        video_count += 1

        if video_count >= MAX_VIDEOS_PER_CHANNEL:
            break

        next_page_token = response.get("nextPageToken")

        if not next_page_token:
            break

In [7]:
len(video_rows)

33332

In [8]:
video_rows[0]

{'channel_id': 'UC0T6MVd3wQDB5ICAe45OxaQ',
 'video_id': '5vB86woZHow',
 'video_title': 'Facebook Ads Have Changed in 2026! Here’s the NEW Strategy',
 'description': "Facebook Ads Have Changed in 2026! Here’s the NEW Strategy\n\n🔴 Become a Performance Marketer in 12 Weeks: 12+ Projects, and 11+ Case Study & More (👉 Apply Today): https://www.wscubetech.com/performance-marketing-course?utm_source=YouTube&utm_medium=YT_Video&utm_campaign=Random_Video&utm_post_id=WsJan2026_05\n\n👉 Join our Official WhatsApp Channel for daily Digital Marketing updates, job alerts & hacks: https://tinyurl.com/marketing-geeks-yt\n\nWsCube Tech is a Vernacular Upskilling platform revolutionizing the way you learn and develop your career skills.🚀\n\nWsCube Tech stands out as a leading EdTech platform, offering comprehensive education in Digital Marketing, Digital Advertising, and various Social Media Marketing skills. Our approach involves both online and classroom training, featuring hands-on projects delivered

In [9]:
# Fetch video stats (Batch API Calls)

video_ids = [row["video_id"] for row in video_rows]

video_stats = {}

for i in range(0, len(video_ids), 50):
    batch_ids = ",".join(video_ids[i:i+50])

    response = youtube.videos().list(part="statistics",id=batch_ids).execute()

    for item in response["items"]:
        stats = item["statistics"]

        video_stats[item["id"]] = {
            "view_count": int(stats.get("viewCount", 0)),
            "like_count": int(stats.get("likeCount", 0)),
            "comment_count": int(stats.get("commentCount", 0))
        }


In [10]:
len(video_stats)

33332

In [11]:
# Merge Stats into Video Rows

for row in video_rows:
    stats = video_stats.get(row["video_id"], {})

    row["view_count"] = stats.get("view_count", 0)
    row["like_count"] = stats.get("like_count", 0)
    row["comment_count"] = stats.get("comment_count", 0)

In [12]:
df = pd.DataFrame(video_rows)

df["published_at"] = pd.to_datetime(df["published_at"])

In [13]:
# Engagement rate calculation (Weighted, View Normalized)

df["weighted_engagement"] = (
    df["like_count"] + 2 * df["comment_count"]
)

df["engagement_rate"] = (
    df["weighted_engagement"] / df["view_count"]
)

df["engagement_rate"] = (
    df["engagement_rate"]
    .replace([float("inf")], 0)
    .fillna(0)
)

df["engagement_rate"] *= 100

In [14]:
# Remove invalid rows
df = df[df["view_count"] > 0]

# Optional: reset index
df.reset_index(drop=True, inplace=True)

In [15]:
# log trsnform subsribers column

df["log_subscriber_count"] = np.log1p(df["subscriber_count"])

In [16]:
# Feature Engineering

df["upload_hour"] = df["published_at"].dt.hour
df["upload_day"] = df["published_at"].dt.dayofweek
df["is_weekend"] = df["upload_day"].isin([5, 6]).astype(int)

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33320 entries, 0 to 33319
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype              
---  ------                --------------  -----              
 0   channel_id            33320 non-null  object             
 1   video_id              33320 non-null  object             
 2   video_title           33320 non-null  object             
 3   description           33320 non-null  object             
 4   published_at          33320 non-null  datetime64[ns, UTC]
 5   subscriber_count      33320 non-null  int64              
 6   view_count            33320 non-null  int64              
 7   like_count            33320 non-null  int64              
 8   comment_count         33320 non-null  int64              
 9   weighted_engagement   33320 non-null  int64              
 10  engagement_rate       33320 non-null  float64            
 11  log_subscriber_count  33320 non-null  float64            
 12  uplo

In [23]:
final_df = df[
    [
        "video_title",
        "description",
        "log_subscriber_count",
        "upload_hour",
        "upload_day",
        "is_weekend",
        "engagement_rate"   # TARGET
    ]
]

In [24]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33320 entries, 0 to 33319
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   video_title           33320 non-null  object 
 1   description           33320 non-null  object 
 2   log_subscriber_count  33320 non-null  float64
 3   upload_hour           33320 non-null  int32  
 4   upload_day            33320 non-null  int32  
 5   is_weekend            33320 non-null  int64  
 6   engagement_rate       33320 non-null  float64
dtypes: float64(2), int32(2), int64(1), object(2)
memory usage: 1.5+ MB


In [25]:
final_df.to_csv(
    "youtube_preupload_engagement_dataset.csv",
    index=False
)